# Notebook 12 — Robustness Checks and Ablations

## Purpose
I verify that the model results are not artefacts of a specific random seed,
preprocessing choice, or data subset.

## Checks
1. Shuffled-label control (should drop to chance)
2. Seed sensitivity (10 seeds)
3. Feature ablation (remove top feature, re-evaluate)
4. Data reduction (train on 50% of training set)
5. Alternative time split

## Why this matters
A result that only holds for seed=42 or a specific preprocessing choice
is fragile.  Robustness checks make the conclusions more credible.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

sys.path.insert(0, str(Path('..').resolve()))
from src.config import load_config
from src.paths import Paths
from src.metrics import macro_f1
from src.utils import set_all_seeds, save_table

cfg   = load_config()
paths = Paths(cfg)

train = pd.read_parquet(paths.processed / 'train.parquet')
test  = pd.read_parquet(paths.processed / 'test.parquet')
FEAT  = cfg['modeling']['feature_cols_review']

train_rv = train.dropna(subset=['review_score'] + FEAT)
test_rv  = test.dropna(subset=['review_score']  + FEAT)
X_tr = train_rv[FEAT].fillna(0).values
y_tr = train_rv['review_score'].astype(int).values
X_te = test_rv[FEAT].fillna(0).values
y_te = test_rv['review_score'].astype(int).values
print("Data ready.")


Data ready.


In [2]:
# Check 1: Shuffled-label control
rng = np.random.default_rng(42)
y_tr_shuffled = rng.permutation(y_tr)

rf_shuf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_shuf.fit(X_tr, y_tr_shuffled)
shuf_acc = accuracy_score(y_te, rf_shuf.predict(X_te))
print(f"Shuffled-label control accuracy: {shuf_acc:.4f}")
print("Expected: near majority-class rate (~0.55 if most reviews are 5-star)")
print("If this is similar to the real model, the model learned nothing useful.")


Shuffled-label control accuracy: 0.5843
Expected: near majority-class rate (~0.55 if most reviews are 5-star)
If this is similar to the real model, the model learned nothing useful.


In [3]:
# Check 2: Seed sensitivity
print("\n=== Seed Sensitivity (10 seeds) ===")
seed_results = []
for seed in range(10):
    rf = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, rf.predict(X_te))
    f1  = macro_f1(y_te, rf.predict(X_te))
    seed_results.append({'seed': seed, 'accuracy': acc, 'macro_f1': f1})
    print(f"  seed={seed}: acc={acc:.4f}  f1={f1:.4f}")

seed_df = pd.DataFrame(seed_results)
print(f"\nAccuracy: mean={seed_df['accuracy'].mean():.4f}  std={seed_df['accuracy'].std():.5f}")
print(f"Macro F1: mean={seed_df['macro_f1'].mean():.4f}  std={seed_df['macro_f1'].std():.5f}")
save_table(seed_df, 'seed_sensitivity',
           reports_dir=str(paths.reports_tabs),
           paper_dir=str(paths.paper_tabs))



=== Seed Sensitivity (10 seeds) ===
  seed=0: acc=0.6173  f1=0.2573
  seed=1: acc=0.6176  f1=0.2573
  seed=2: acc=0.6176  f1=0.2584
  seed=3: acc=0.6187  f1=0.2566
  seed=4: acc=0.6173  f1=0.2569
  seed=5: acc=0.6177  f1=0.2579
  seed=6: acc=0.6169  f1=0.2547
  seed=7: acc=0.6175  f1=0.2560
  seed=8: acc=0.6185  f1=0.2577
  seed=9: acc=0.6181  f1=0.2555

Accuracy: mean=0.6177  std=0.00057
Macro F1: mean=0.2568  std=0.00115
  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\reports\tables\seed_sensitivity.csv
  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\paper_or_report\tables\seed_sensitivity.csv


[WindowsPath('C:/Users/Peter/Documents/projects/Jobberman_projects/double_Integral/ecommerce_customer_intelligence/reports/tables/seed_sensitivity.csv'),
 WindowsPath('C:/Users/Peter/Documents/projects/Jobberman_projects/double_Integral/ecommerce_customer_intelligence/paper_or_report/tables/seed_sensitivity.csv')]

In [4]:
# Check 3: Feature ablation — remove top 1 feature
import joblib
rf_full = joblib.load(paths.models / 'rf_review_model.joblib')
importances = rf_full.feature_importances_
top_feat_idx = np.argmax(importances)
top_feat_name = FEAT[top_feat_idx]

print(f"\n=== Feature Ablation: removing '{top_feat_name}' ===")
FEAT_ABLATED = [f for i, f in enumerate(FEAT) if i != top_feat_idx]
X_tr_abl = train_rv[FEAT_ABLATED].fillna(0).values
X_te_abl = test_rv[FEAT_ABLATED].fillna(0).values

rf_abl = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_abl.fit(X_tr_abl, y_tr)
abl_acc = accuracy_score(y_te, rf_abl.predict(X_te_abl))
full_acc = accuracy_score(y_te, rf_full.predict(X_te))
print(f"Full model accuracy:    {full_acc:.4f}")
print(f"Ablated model accuracy: {abl_acc:.4f}")
print(f"Drop from removing top feature: {(full_acc - abl_acc)*100:.2f} pp")



=== Feature Ablation: removing 'delivery_delay_days' ===
Full model accuracy:    0.5493
Ablated model accuracy: 0.5894
Drop from removing top feature: -4.02 pp


In [5]:
# Check 4: Data reduction (50% of training data)
print("\n=== Data Reduction (50% training set) ===")
rng = np.random.default_rng(42)
idx_50 = rng.choice(len(X_tr), size=len(X_tr)//2, replace=False)
X_tr_50 = X_tr[idx_50]
y_tr_50 = y_tr[idx_50]

rf_50 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_50.fit(X_tr_50, y_tr_50)
acc_50 = accuracy_score(y_te, rf_50.predict(X_te))
print(f"Full training size: {len(X_tr):,}  Accuracy: {full_acc:.4f}")
print(f"50%  training size: {len(X_tr_50):,}  Accuracy: {acc_50:.4f}")
print(f"Performance with 50% data: {acc_50/full_acc*100:.1f}% of full-data performance")



=== Data Reduction (50% training set) ===
Full training size: 42,597  Accuracy: 0.5493
50%  training size: 21,298  Accuracy: 0.6209
Performance with 50% data: 113.0% of full-data performance


In [6]:
# Summary
robustness_summary = pd.DataFrame([
    {'Check': 'Full model (baseline)', 'Accuracy': full_acc},
    {'Check': 'Shuffled-label control', 'Accuracy': shuf_acc},
    {'Check': 'Seed sensitivity (mean)', 'Accuracy': seed_df['accuracy'].mean()},
    {'Check': f'Ablated ({top_feat_name} removed)', 'Accuracy': abl_acc},
    {'Check': '50% training data', 'Accuracy': acc_50},
])
robustness_summary['Accuracy'] = robustness_summary['Accuracy'].round(4)
print("=== Robustness Summary ===")
print(robustness_summary.to_string(index=False))
save_table(robustness_summary, 'robustness_checks',
           reports_dir=str(paths.reports_tabs),
           paper_dir=str(paths.paper_tabs))
print("\nNotebook 12 complete.")


=== Robustness Summary ===
                                Check  Accuracy
                Full model (baseline)    0.5493
               Shuffled-label control    0.5843
              Seed sensitivity (mean)    0.6177
Ablated (delivery_delay_days removed)    0.5894
                    50% training data    0.6209
  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\reports\tables\robustness_checks.csv
  Saved table: C:\Users\Peter\Documents\projects\Jobberman_projects\double_Integral\ecommerce_customer_intelligence\paper_or_report\tables\robustness_checks.csv

Notebook 12 complete.
